# 04 — Sieve Constraints

**Repo:** `github.com/thinkthoughts/prime-numbers-lab`  
**Purpose:** measure sieve filtering as layered divisibility constraints.

Notebook 01 measured a finite residue constraint.  
Notebook 02 measured local gap structure.  
Notebook 03 measured global density structure.  
Notebook 04 measures the mechanism of layered filtering.

\[
S_k = S_{k-1} \setminus \{n : q_k \mid n,\ n \ne q_k\}
\]

The sieve does not guess primes. It removes invalid divisibility assignments until prime structure remains under layered constraint.

## 0. Setup

Artifact structure:

```text
04_sieve_constraints/
├── data/
├── docs/
├── figures/
└── tex/
```

Root export:

```text
04_sieve_constraints_export.zip
```

In [ ]:
from pathlib import Path
import json
import math
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

NOTEBOOK_ID = "04_sieve_constraints"
NOTEBOOK_NUM = NOTEBOOK_ID.split("_")[0]
NOTEBOOK_TITLE = "Sieve Constraints"

OUT = Path(NOTEBOOK_ID)
DATA_DIR = OUT / "data"
DOCS_DIR = OUT / "docs"
FIG_DIR = OUT / "figures"
TEX_DIR = OUT / "tex"

for d in [DATA_DIR, DOCS_DIR, FIG_DIR, TEX_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Artifact directory: {OUT.resolve()}")

## 1. Premise

The sieve gives the strongest recoverability example so far.

- **Remains under constraint / persists:** candidate values remain after divisibility filters.
- **Drift:** invalid candidates are removed by each filter.
- **Recoverability:** applying enough prime filters recovers prime identity exactly up to \(N\).

Unlike modulo 6, the full sieve is sufficient when filters are applied through \(\sqrt{N}\).

## 2. Constraint definition

Start with:

\[
S_0 = \{2,3,4,\dots,N\}
\]

Let \(q_k\) be the \(k\)-th prime filter.

Each sieve layer removes composite multiples of \(q_k\):

\[
S_k = S_{k-1} \setminus \{n : q_k \mid n,\ n \ne q_k\}
\]

The final retained set is:

\[
S_{\mathrm{final}} = \{p : p \le N,\ p \text{ prime}\}
\]

when all prime filters \(q_k \le \sqrt{N}\) are applied.

In [ ]:
# Parameters

N_MAX = 200_000
RANDOM_SEED = 9423

params = {
    "N_MAX": N_MAX,
    "RANDOM_SEED": RANDOM_SEED,
    "NOTEBOOK_ID": NOTEBOOK_ID,
    "NOTEBOOK_TITLE": NOTEBOOK_TITLE,
    "constraint": "layered sieve filtering by prime divisibility",
}

params

## 3. Baseline prime generator

Use a standard sieve to produce the reference prime set.  
Then run a layer-by-layer sieve to record removals.

In [ ]:
def simple_sieve(n: int) -> np.ndarray:
    if n < 2:
        return np.array([], dtype=int)
    s = np.ones(n + 1, dtype=bool)
    s[:2] = False
    for i in range(2, int(math.sqrt(n)) + 1):
        if s[i]:
            s[i*i:n+1:i] = False
    return np.nonzero(s)[0]

reference_primes = simple_sieve(N_MAX)
filter_primes = reference_primes[reference_primes <= int(math.sqrt(N_MAX))]

summary = {
    "n_max": int(N_MAX),
    "reference_prime_count": int(len(reference_primes)),
    "filter_prime_count_leq_sqrt_n": int(len(filter_primes)),
    "sqrt_n": float(math.sqrt(N_MAX)),
    "first_primes": reference_primes[:10].tolist(),
    "last_primes": reference_primes[-10:].tolist(),
    "filter_primes": filter_primes.tolist(),
}

summary

## 4. Layer-by-layer sieve measurement

Track:

- retained candidate count
- removed count by layer
- retention share
- drift share

In [ ]:
# Candidate universe starts at 2..N.
candidate_mask = np.zeros(N_MAX + 1, dtype=bool)
candidate_mask[2:] = True

initial_count = int(candidate_mask.sum())
rows = []

for layer_index, q in enumerate(filter_primes, start=1):
    before_count = int(candidate_mask.sum())

    # Remove composite multiples of q, preserving q itself.
    remove_mask = candidate_mask.copy()
    remove_mask[:q+1] = False
    remove_mask &= (np.arange(N_MAX + 1) % q == 0)

    removed_count = int(remove_mask.sum())
    candidate_mask[remove_mask] = False
    after_count = int(candidate_mask.sum())

    rows.append({
        "layer": layer_index,
        "filter_prime_q": int(q),
        "candidate_count_before": before_count,
        "removed_count": removed_count,
        "candidate_count_after": after_count,
        "retention_share_initial": after_count / initial_count,
        "removed_share_initial": (initial_count - after_count) / initial_count,
        "removed_share_layer": removed_count / before_count if before_count else 0.0,
    })

sieve_layers_df = pd.DataFrame(rows)

sieve_primes = np.nonzero(candidate_mask)[0]
sieve_primes = sieve_primes[sieve_primes >= 2]

exact_match = np.array_equal(sieve_primes, reference_primes)

sieve_layers_df.head(), exact_match

## 5. CGCS score and drift

For this notebook, exact recovery score is:

\[
CGCS_{\mathrm{sieve}} =
\frac{|S_{\mathrm{final}} \cap P_N|}{|P_N|}
\]

where \(P_N = \{p : p \le N\}\).

For a correct full sieve:

\[
CGCS_{\mathrm{sieve}} = 1
\]

The layer-level drift is:

\[
drift_k = 1 - \frac{|S_k|}{|S_0|}
\]

In [ ]:
reference_set = set(reference_primes.tolist())
sieve_set = set(sieve_primes.tolist())

true_positive_count = len(reference_set & sieve_set)
false_positive_count = len(sieve_set - reference_set)
false_negative_count = len(reference_set - sieve_set)

cgcs_sieve = true_positive_count / len(reference_set) if reference_set else float("nan")
precision_sieve = true_positive_count / len(sieve_set) if sieve_set else float("nan")
recall_sieve = cgcs_sieve

final_retention_share = len(sieve_primes) / initial_count
final_drift_share = 1.0 - final_retention_share

measurement = {
    "initial_candidate_count": int(initial_count),
    "final_candidate_count": int(len(sieve_primes)),
    "reference_prime_count": int(len(reference_primes)),
    "true_positive_count": int(true_positive_count),
    "false_positive_count": int(false_positive_count),
    "false_negative_count": int(false_negative_count),
    "exact_match_reference_primes": bool(exact_match),
    "cgcs_sieve": float(cgcs_sieve),
    "precision_sieve": float(precision_sieve),
    "recall_sieve": float(recall_sieve),
    "final_retention_share": float(final_retention_share),
    "final_drift_share": float(final_drift_share),
}

cgcs = {
    "score": float(cgcs_sieve),
    "definition": "CGCS_sieve = |S_final ∩ P_N| / |P_N| after filters q <= sqrt(N)",
    "interpretation": "1.0 means the layered sieve recovers every prime up to N.",
}

measurement

## 6. Recoverability note

This notebook has a stronger recoverability claim than modulo 6.

Modulo 6 recovers candidate classes.

The full sieve recovers prime identity exactly up to \(N\), provided all prime filters \(q \le \sqrt{N}\) are applied.

In [ ]:
recoverability = {
    "recoverability_score_exact_prime_identity": float(precision_sieve * recall_sieve),
    "definition": "precision × recall against reference prime set",
    "note": "full sieve recovers prime identity exactly up to N when all prime filters q <= sqrt(N) are applied",
}

recoverability

## 7. Figure 1 — retained candidates by layer

Candidate count decreases as divisibility constraints are applied.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(sieve_layers_df["layer"], sieve_layers_df["candidate_count_after"], marker="o")
ax.set_title("Retained candidates by sieve layer")
ax.set_xlabel("sieve layer")
ax.set_ylabel("candidate count after layer")
ax.grid(True, alpha=0.3)

fig1_path = FIG_DIR / f"{NOTEBOOK_NUM}_retained_candidates_by_layer.png"
fig.savefig(fig1_path, dpi=180, bbox_inches="tight")
plt.show()

fig1_path

## 8. Figure 2 — removed count by prime filter

Early filters remove more candidates; later filters remove fewer because many composites have already drifted out.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(sieve_layers_df["filter_prime_q"], sieve_layers_df["removed_count"])
ax.set_title("Removed candidates by prime filter")
ax.set_xlabel("filter prime q")
ax.set_ylabel("removed count")
ax.grid(True, axis="y", alpha=0.3)

fig2_path = FIG_DIR / f"{NOTEBOOK_NUM}_removed_by_filter.png"
fig.savefig(fig2_path, dpi=180, bbox_inches="tight")
plt.show()

fig2_path

## 9. Figure 3 — retention and drift by layer

Retention decreases as filtering proceeds.  
Drift increases because invalid candidates are removed.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(sieve_layers_df["layer"], sieve_layers_df["retention_share_initial"], marker="o", label="retention share")
ax.plot(sieve_layers_df["layer"], sieve_layers_df["removed_share_initial"], marker="o", label="drift share")
ax.set_title("Retention and drift by sieve layer")
ax.set_xlabel("sieve layer")
ax.set_ylabel("share of initial candidates")
ax.legend()
ax.grid(True, alpha=0.3)

fig3_path = FIG_DIR / f"{NOTEBOOK_NUM}_retention_and_drift_by_layer.png"
fig.savefig(fig3_path, dpi=180, bbox_inches="tight")
plt.show()

fig3_path

## 10. Interpretation

1. **What remains under constraint?**  
   Candidate values that are not removed by prime divisibility filters.

2. **What drifts?**  
   Composite multiples drift out layer by layer.

3. **What is recoverable?**  
   Prime identity is exactly recoverable up to \(N\) when all filters \(q \le \sqrt{N}\) are applied.

4. **What should not be overclaimed?**  
   This is finite exact recovery by a known algorithm, not a proof of a new prime theorem.

In [ ]:
interpretation_lines = [
    f"# {NOTEBOOK_TITLE}",
    "",
    "## Constraint result",
    "",
    "This notebook measured sieve filtering as layered divisibility constraints.",
    "",
    f"The candidate universe started with {initial_count:,} integers from 2 to {N_MAX:,}.",
    f"After applying all prime filters q <= sqrt(N), the final retained count was {len(sieve_primes):,}.",
    "",
    "## Remains under constraint",
    "",
    "Values that remain after all divisibility filters match the reference prime set.",
    "",
    "## Drift",
    "",
    "Composite multiples drift out layer by layer. Early filters remove the largest number of candidates.",
    "",
    "## CGCS score",
    "",
    "CGCS_sieve = |S_final ∩ P_N| / |P_N|.",
    "",
    f"- CGCS_sieve = {cgcs_sieve:.6f}",
    f"- precision = {precision_sieve:.6f}",
    f"- recall = {recall_sieve:.6f}",
    f"- exact match = {exact_match}",
    "",
    "## Recoverability",
    "",
    "The full sieve recovers prime identity exactly up to N when all prime filters q <= sqrt(N) are applied.",
    "",
    "## Caution",
    "",
    "This notebook demonstrates finite exact recovery by the classical sieve. It does not claim a new primality theorem.",
]

interpretation = "\n".join(interpretation_lines)

figure_paths = [fig1_path, fig2_path, fig3_path]
figure_titles = [
    "Retained candidates by sieve layer",
    "Removed candidates by prime filter",
    "Retention and drift by layer",
]

figures_md = "\n\n## Figures\n\n"
for i, (fig, title) in enumerate(zip(figure_paths, figure_titles), start=1):
    figures_md += f"### Figure {i} — {title}\n\n"
    figures_md += f"![Figure {i}](../figures/{fig.name})\n\n"

print(interpretation + figures_md)

## 11. Export data, notes, math, and TeX

In [ ]:
summary_df = pd.DataFrame([{
    **params,
    **summary,
    **measurement,
    **recoverability,
    "cgcs_score": cgcs["score"],
    "cgcs_definition": cgcs["definition"],
}])

summary_path = DATA_DIR / f"{NOTEBOOK_NUM}_summary.csv"
layers_path = DATA_DIR / f"{NOTEBOOK_NUM}_sieve_layers.csv"
primes_path = DATA_DIR / f"{NOTEBOOK_NUM}_sieve_primes.csv"
metadata_path = DATA_DIR / f"{NOTEBOOK_NUM}_metadata.json"

interpretation_path = DOCS_DIR / f"{NOTEBOOK_NUM}_interpretation.md"
design_path = DOCS_DIR / f"{NOTEBOOK_NUM}_design_notes.md"

summary_tex_path = TEX_DIR / f"{NOTEBOOK_NUM}_summary_snippet.tex"
math_tex_path = TEX_DIR / f"{NOTEBOOK_NUM}_math_notes.tex"

summary_df.to_csv(summary_path, index=False)
sieve_layers_df.to_csv(layers_path, index=False)
pd.DataFrame({"prime": sieve_primes}).to_csv(primes_path, index=False)

metadata = {
    "params": params,
    "summary": summary,
    "measurement": measurement,
    "recoverability": recoverability,
    "cgcs": cgcs,
    "figures": [str(p) for p in figure_paths],
    "data": {
        "summary": str(summary_path),
        "sieve_layers": str(layers_path),
        "sieve_primes": str(primes_path),
    },
    "docs": {
        "interpretation": str(interpretation_path),
        "design_notes": str(design_path),
    },
    "tex": {
        "summary_snippet": str(summary_tex_path),
        "math_notes": str(math_tex_path),
    },
}

metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
interpretation_path.write_text(interpretation + figures_md + "\n", encoding="utf-8")

design_lines = [
    f"# Design Notes — {NOTEBOOK_TITLE}",
    "",
    "## Notebook role",
    "",
    "Notebook 04 follows Notebook 03 by moving from density measurement to the finite filtering mechanism that recovers primes.",
    "",
    "Notebook 01 measured residue constraints.",
    "Notebook 02 measured gap structure.",
    "Notebook 03 measured density scale.",
    "Notebook 04 measures layered sieve filtering.",
    "",
    "## Constraint",
    "",
    "Each layer removes composite multiples of a prime filter q:",
    "",
    "S_k = S_(k-1) \\ {n : q_k divides n, n != q_k}.",
    "",
    "## Measurement",
    "",
    "The notebook computes:",
    "",
    "1. retained candidate count by layer",
    "2. removed candidate count by filter",
    "3. retention share",
    "4. drift share",
    "5. exact recovery against the reference prime set",
    "",
    "## CGCS score",
    "",
    "CGCS_sieve = |S_final ∩ P_N| / |P_N| after filters q <= sqrt(N).",
    "",
    "Expected result: exactly 1.0.",
    "",
    "## Figures",
    "",
    "1. retained candidates by sieve layer",
    "2. removed candidates by prime filter",
    "3. retention and drift by layer",
    "",
    "## Recoverability",
    "",
    "The full sieve recovers prime identity exactly up to N when all prime filters q <= sqrt(N) are applied.",
    "",
    "## Handoff",
    "",
    "Notebook 05 should compare prime structure against random candidate sets.",
]

design_path.write_text("\n".join(design_lines) + "\n", encoding="utf-8")

summary_tex_lines = [
    rf"\section*{{{NOTEBOOK_TITLE}}}",
    "",
    r"This notebook measures layered sieve filtering:",
    r"\[",
    r"S_k = S_{k-1} \setminus \{n : q_k \mid n,\ n \ne q_k\}.",
    r"\]",
    "",
    rf"For $N={N_MAX:,}$:",
    r"\begin{itemize}",
    rf"  \item initial candidates $= {initial_count}$",
    rf"  \item final retained candidates $= {len(sieve_primes)}$",
    rf"  \item reference prime count $= {len(reference_primes)}$",
    rf"  \item $CGCS_{{sieve}} = {cgcs_sieve:.6f}$",
    rf"  \item exact match $= {exact_match}$",
    r"\end{itemize}",
    "",
    r"The full sieve recovers prime identity exactly up to $N$ when all prime filters $q \le \sqrt{N}$ are applied.",
]

summary_tex_path.write_text("\n".join(summary_tex_lines) + "\n", encoding="utf-8")

math_tex_lines = [
    r"\documentclass{article}",
    r"\usepackage{amsmath}",
    r"\usepackage{amssymb}",
    r"\usepackage[margin=1in]{geometry}",
    "",
    r"\begin{document}",
    "",
    r"\section*{Math Notes: Sieve Constraints}",
    "",
    r"\subsection*{Initial candidate set}",
    "",
    r"\[",
    r"S_0 = \{2,3,4,\dots,N\}.",
    r"\]",
    "",
    r"\subsection*{Layered sieve update}",
    "",
    r"Let $q_k$ denote the $k$-th prime filter.",
    r"\[",
    r"S_k = S_{k-1} \setminus \{n : q_k \mid n,\ n \ne q_k\}.",
    r"\]",
    "",
    r"\subsection*{Final retained set}",
    "",
    r"If all prime filters $q_k \le \sqrt{N}$ are applied, then",
    r"\[",
    r"S_{\mathrm{final}} = \{p : p \le N,\ p \text{ prime}\}.",
    r"\]",
    "",
    r"\subsection*{Retention and drift}",
    "",
    r"\[",
    r"retention_k = \frac{|S_k|}{|S_0|}.",
    r"\]",
    r"\[",
    r"drift_k = 1 - retention_k.",
    r"\]",
    "",
    r"\subsection*{CGCS score}",
    "",
    r"Let $P_N = \{p : p \le N\}$.",
    r"\[",
    r"CGCS_{\mathrm{sieve}} =",
    r"\frac{|S_{\mathrm{final}} \cap P_N|}{|P_N|}.",
    r"\]",
    "",
    r"Measured value:",
    r"\[",
    rf"CGCS_{{\mathrm{{sieve}}}} = {cgcs_sieve:.6f}.",
    r"\]",
    "",
    r"\subsection*{Recoverability}",
    "",
    r"The full sieve recovers prime identity exactly up to $N$, provided all filters $q \le \sqrt{N}$ are applied.",
    "",
    r"\end{document}",
]

math_tex_path.write_text("\n".join(math_tex_lines) + "\n", encoding="utf-8")

summary_path, layers_path, primes_path, metadata_path, interpretation_path, design_path, summary_tex_path, math_tex_path

## 12. Export zip

Pi-stage-lab style root export zip, with optional Colab download lines left commented.

In [ ]:
EXPORT_NAME = f"{NOTEBOOK_ID}_export.zip"

with zipfile.ZipFile(EXPORT_NAME, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in [DOCS_DIR, DATA_DIR, FIG_DIR, TEX_DIR]:
        for path in folder.rglob("*"):
            if path.is_file():
                z.write(path, path.as_posix())

print(f"Export ready: {EXPORT_NAME}")
print("Tip: uncomment Colab lines below to download.")

# --- Optional Colab download ---
# Uncomment the lines below when running in Colab
#
# from google.colab import files
# files.download(EXPORT_NAME)

## 13. Next notebook handoff

Next notebook:

```text
05_random_vs_prime.ipynb
```

Purpose:

> compare prime structure against random candidate sets to show that constraint-retained structure differs from unconstrained sampling.

In [ ]:
next_step = "Notebook 05: random versus prime structure."
print(next_step)